In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-5"


### Making my first request

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

print(message.content[0].text)


### Multi- Turn Conversations 
### Building helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# putting into practice 

messages = []

add_user_message(messages, "Define quantum computing in one sentence")

answer = chat(messages)

add_assistant_message(messages, answer)

add_user_message(messages, "Write Another Sentence")

final_answer = chat(messages)



### Lesson 5 System prompts
# building a flexible cha fucntion

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }


    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Now you can call the chat function with or without a system promt:

# Without system prompt
answer = chat(messages)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)

print(answer)




Quantum computing is a type of computation that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.
Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are strictly either 0 or 1.


### Project Overview

We're going to build a practical project that teaches Claude how to set reminders for future dates. This might sound simple at first, but it reveals several interesting challenges that we'll solve using custom tools.

#### Tool functions

When building AI applications with Claude, you'll often need to give it access to real-time information or the ability to perform actions. This is where tool functions come in - they're Python functions that Claude can call when it needs additional data to help users.

In [17]:
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [19]:
# Default format: "2024-01-15 14:30:25"
print(get_current_datetime())
print(get_current_datetime("%Y"))


2026-09-16 17:21:21
2026


### Tool JSON schemas

After writing your tool function, the next step is creating a JSON schema that tells Claude what arguments your function expects and how to use it. This schema acts as documentation that Claude reads to understand when and how to call your tools.

In [22]:
from anthropic.types import ToolParam


get_current_datetime_schema = ToolParam( {
    "name": "get_current_datetime",
    "description": "Get the current date and time. Use this whenever you need to know today's date, the current time, or the day of the week.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A Python strftime format string, e.g. '%Y-%m-%d %H:%M:%S' for '2026-09-16 14:30:25' or '%A, %d %B %Y' for 'Wednesday, 16 September 2026'. Defaults to '%Y-%m-%d %H:%M:%S'.",
            }
        },
        "required": [],
    },
})

#### Handling Message Blocks

When working with Claude's tool functionality, you'll encounter a new type of response structure that's different from the simple text responses you've seen before. Instead of just getting back a single text block, Claude can now return multi-block messages that contain both text and tool usage information.

In [24]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)


print(response)

Message(id='msg_011Cf7S9xD8rqCZDmaEttRYm', container=None, content=[ToolUseBlock(id='toolu_016PuD6DKQMzajuKnDh4KBfh', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=680, output_tokens=62, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


The tool parameter takes the json schema tools claude can call upon

In [25]:
messages.append({
    "role": "assistant",
    "content": response.content
})

In [32]:
## Full loop as described
messages = [{"role": "user", "content": "What day of the week is it?"}]

# 1-2. Claude decides to use the tool this is an API call
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print(response)

# Preserve the whole block list
messages.append({"role": "assistant", "content": response.content})
print(response)


# 3. Find the tool call and run the real function
tool_results = []
for block in response.content:
    if block.type == "tool_use":
        output = get_current_datetime(**block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": output,
        })

# 4. Send results back as a USER message
messages.append({"role": "user", "content": tool_results})

# 5. Final answer
final = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)
print(messages)

Message(id='msg_011Cf7TifsAPZWH6sdStYaMy', container=None, content=[ToolUseBlock(id='toolu_01Pi2Z9rSSEFu8mQa84Youoa', caller=DirectCaller(type='direct'), input={'date_format': '%A'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=673, output_tokens=58, output_tokens_details=None, server_tool_use=None, service_tier='standard'))
Message(id='msg_011Cf7TifsAPZWH6sdStYaMy', container=None, content=[ToolUseBlock(id='toolu_01Pi2Z9rSSEFu8mQa84Youoa', caller=DirectCaller(type='direct'), input={'date_format': '%A'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-sonnet-4-5-20250929', role='assistant', stop_details

In [31]:
for block in response.content:
    print(f"type={block.type}")
    if block.type == "tool_use":
        print(f"  id={block.id}")
        print(f"  input={block.input}")

type=tool_use
  id=toolu_016CH5gLQxnshTTfbUX5ZTS4
  input={'date_format': '%A'}


### Multi-turn conversations with tools


Claude might need to handle 6 tools in sequence

In [42]:
from anthropic.types import Message


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, stop_sequences=None, tools=None, max_tokens=4000):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools

    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

In [ ]:
def run_conversation(messages, tools, max_turns=10):
    for _ in range(max_turns):
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            return messages

        add_user_message(messages, run_tools(response))

    return messages

In [44]:
messages = []
add_user_message(messages, "What day of the week is it?")
messages = run_conversation(messages, [get_current_datetime_schema])
print(messages[-1]["content"][0].text)

It's **Wednesday** today.


### Implementing Multi turns
Building a conversation system with tools requires implementing a loop that keeps calling Claude until it stops requesting tool usage. When Claude no longer asks for tools, that signals it has a final response ready for the user.

In [ ]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages


### THis continures until claude finds an anser without tools

### Handling multiple tool calls

In [48]:
import json


def run_tools(message):
    tool_requests = [b for b in message.content if b.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        tool_result_blocks.append(block)

    return tool_result_blocks

In [49]:
from datetime import datetime, timedelta


def add_duration_to_datetime(datetime_str, duration=0, unit="days",
                             input_format="%Y-%m-%d %H:%M:%S"):
    dt = datetime.strptime(datetime_str, input_format)

    if unit == "days":
        dt += timedelta(days=duration)
    elif unit == "hours":
        dt += timedelta(hours=duration)
    elif unit == "minutes":
        dt += timedelta(minutes=duration)
    else:
        raise ValueError(f"Unsupported unit: {unit}")

    return dt.strftime("%A, %B %d, %Y %H:%M:%S")

In [50]:
add_duration_schema = {
    "name": "add_duration_to_datetime",
    "description": "Add a duration to a datetime string and return the resulting datetime. Use this for questions about future or past dates.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {"type": "string", "description": "Starting datetime, format '%Y-%m-%d %H:%M:%S'"},
            "duration": {"type": "number", "description": "Amount to add"},
            "unit": {"type": "string", "enum": ["days", "hours", "minutes"]},
        },
        "required": ["datetime_str"],
    },
}

In [51]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")

In [53]:
def run_conversation(messages, tools, max_turns=10):
    for _ in range(max_turns):
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        text = text_from_message(response)
        if text:
            print(text)

        if response.stop_reason != "tool_use":
            return messages

        add_user_message(messages, run_tools(response))

    print(f"Stopped after {max_turns} turns")
    return messages

In [54]:
messages = []
add_user_message(messages, "What day of the week is it in 90 days?")
run_conversation(messages, [get_current_datetime_schema, add_duration_schema])

I'll help you find out what day of the week it will be in 90 days.

First, let me get the current date and time, then add 90 days to it.
In 90 days, it will be **Tuesday, December 15, 2026**.


[{'role': 'user', 'content': 'What day of the week is it in 90 days?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you find out what day of the week it will be in 90 days.\n\nFirst, let me get the current date and time, then add 90 days to it.", type='text'),
   ToolUseBlock(id='toolu_01H45YdtanBbLooCPcfDMyXv', caller=DirectCaller(type='direct'), input={'date_format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01H45YdtanBbLooCPcfDMyXv',
    'content': '"2026-09-16 20:42:55"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01GNbRxXykDoZCT7YHK19oUz', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-09-16 20:42:55', 'duration': 90, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result'

#### Using multiple tools and adding a reminder

In [60]:
### Update run_converation 
def run_conversation(messages, tools, max_turns=10, fine_grained=True):
    for _ in range(max_turns):
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        text = text_from_message(response)
        if text:
            print(text)

        if response.stop_reason != "tool_use":
            return messages

        add_user_message(messages, run_tools(response))

    print(f"Stopped after {max_turns} turns")
    return messages


In [56]:
##Updating the tool router

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")

In [57]:
import json
from datetime import datetime
from pathlib import Path

REMINDERS_FILE = Path("reminders.json")


def set_reminder(content, timestamp):
    """Store a reminder with its due time."""
    if not content:
        raise ValueError("content cannot be empty")

    # Validate the timestamp now rather than failing later
    due = datetime.fromisoformat(timestamp)

    reminders = []
    if REMINDERS_FILE.exists():
        reminders = json.loads(REMINDERS_FILE.read_text())

    reminder = {
        "content": content,
        "timestamp": due.isoformat(),
        "created": datetime.now().isoformat(),
    }
    reminders.append(reminder)
    REMINDERS_FILE.write_text(json.dumps(reminders, indent=2))

    return f"Reminder set for {due.strftime('%A, %d %B %Y at %H:%M')}: {content}"

In [58]:
set_reminder_schema = {
    "name": "set_reminder",
    "description": "Set a reminder for the user at a specific date and time. Use this when the user asks to be reminded about something.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "What the reminder is about, in the user's own words.",
            },
            "timestamp": {
                "type": "string",
                "description": "When the reminder is due, as an ISO 8601 string, e.g. '2026-09-20T14:30:00'. Call get_current_datetime first if you need to work out a relative time like 'tomorrow'.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

In [61]:
messages = []
add_user_message(messages, "Remind me to push my git branch in 2 hours")
run_conversation(messages, [get_current_datetime_schema, add_duration_schema, set_reminder_schema])

I'll set a reminder for you to push your git branch in 2 hours. Let me first get the current time and then calculate when that will be.
Perfect! I've set a reminder for you to push your git branch at 22:56 (10:56 PM) today, which is in 2 hours.


[{'role': 'user', 'content': 'Remind me to push my git branch in 2 hours'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll set a reminder for you to push your git branch in 2 hours. Let me first get the current time and then calculate when that will be.", type='text'),
   ToolUseBlock(id='toolu_01A6E8qcAdPn5wnYaY34GcJS', caller=DirectCaller(type='direct'), input={'date_format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01A6E8qcAdPn5wnYaY34GcJS',
    'content': '"2026-09-16 20:56:07"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_016KFKcSArsa8pCZ6W7W7vuP', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-09-16 20:56:07', 'duration': 2, 'unit': 'hours'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result

### Text edit tool

In [62]:
def get_text_edit_schema(model):
    if model.startswith("claude-3-7-sonnet"):
        return {
            "type": "text_editor_20250124",
            "name": "str_replace_editor",
        }
    elif model.startswith("claude-3-5-sonnet"):
        return {
            "type": "text_editor_20241022", 
            "name": "str_replace_editor",
        }